# Phase 3: Constraint-Aware Multi-Objective Bayesian Optimization (Constrained MOBO)
This notebook optimizes the 200 MeV electron injector linac across 3 objectives (\(arepsilon_{nx}, arepsilon_{ny}, \sigma_E\))
while explicitly modeling beam quality constraints (\(\sigma_x, \sigma_y, \sigma_{xp}, \sigma_{yp}, \sigma_z \le 1.0	ext{ mm/mrad}\) and \(195	ext{ MeV} \le E_{	ext{kin}} \le 205	ext{ MeV}\))
using 9 Gaussian Process surrogates and BoTorch constrained \(q	ext{LogNEHVI}\) acquisition.

In [ ]:
# Environment Setup & Imports
%load_ext autoreload
%autoreload 2
import os, sys, time
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# Set PyTorch default dtype
torch.set_default_dtype(torch.double)

# Ensure project root is in sys.path
sys.path.append("..")
from run_astra import run_astra_simulation
from mobo_utils import evaluate_constrained_objective, compute_ref_point
from file_io import create_run_directory, save_results, save_checkpoint, load_checkpoint
from plot_utils import plot_hypervolume, plot_pareto_objective_space, plot_all_constraints, plot_objective_evolution

from botorch.models import SingleTaskGP, ModelListGP
from botorch.models.transforms.input import Normalize
from botorch.models.transforms.outcome import Standardize
from botorch.fit import fit_gpytorch_mll
from gpytorch.mlls import ExactMarginalLogLikelihood
from botorch.optim import optimize_acqf
from botorch.acquisition.multi_objective.logei import qLogNoisyExpectedHypervolumeImprovement
from botorch.acquisition.multi_objective.objective import IdentityMCMultiOutputObjective
from botorch.utils.multi_objective.box_decompositions.non_dominated import FastNondominatedPartitioning
from botorch.utils.multi_objective.hypervolume import Hypervolume
from botorch.utils.multi_objective.pareto import is_non_dominated


In [ ]:
# ── Simulation Configuration Summary ─────────────────────────────────────────
# Displays key parameters for this notebook run in tabular format.
# Defaults reflect original phase3_constrained_mobo configuration.
# 'This Session' shows any overrides applied for the current run.

import pandas as pd

# --------------------------------------------------------------------------
# Notebook-level overrides (edit these to reconfigure a run)
# --------------------------------------------------------------------------
_N_ITERATIONS  = 20    # default: 20
_BATCH_SIZE    = 8     # default:  8 (q)
_INIT_SAMPLES  = 16    # default: 16
_NUM_WORKERS   = 12    # default: 12 (ThreadPoolExecutor)
_SEED          = 42    # default: 42
_ACQ_FN        = "qLogNEHVI (constrained)"  # default
_N_GP_MODELS   = 9    # 3 objectives + 6 constraint diagnostics
_CONSTRAINTS   = ["sigma_x<=1mm", "sigma_y<=1mm", "sigma_xp<=1mrad",
                   "sigma_yp<=1mrad", "sigma_z<=1mm",
                   "E_kin in [195,205] MeV"]  # default
_BOUND_RATIO   = [0.50, 0.50, 0.50, 0.10, 0.10, 0.10]  # ±ratio around nominal

# --------------------------------------------------------------------------
# Build summary table
# --------------------------------------------------------------------------
_rows = [
    ("BO iterations",     "20",   str(_N_ITERATIONS),          "total iterations"),
    ("Batch size (q)",    "8",    str(_BATCH_SIZE),            "candidates/iter"),
    ("Initial samples",  "16",    str(_INIT_SAMPLES),          "Sobol random init"),
    ("Parallel workers", "12",    str(_NUM_WORKERS),           "ThreadPoolExecutor"),
    ("Random seed",      "42",    str(_SEED),                  "reproducibility"),
    ("GP surrogate count","9",    str(_N_GP_MODELS),           "3 obj + 6 constraint GPs"),
    ("Surrogate model",  "ModelListGP", "ModelListGP",         "Matérn-5/2 ARD per GP"),
    ("Acquisition fn",   "qLogNEHVI (constrained)", _ACQ_FN,  "feasibility-weighted"),
    ("Objectives",       "ex, ey, sigma_E", "ex, ey, sigma_E","minimise all 3"),
    ("Design variables", "6D", "6D", "sol, Q1, Q2, phi_gun, phi12, phi34"),
    ("Bound ratio",      "[0.50,0.50,0.50,0.10,0.10,0.10]",
                           str(_BOUND_RATIO),                  "±ratio around astra.in nominal"),
    ("Constraints",
                          "6 beam-quality constraints",
                          str(len(_CONSTRAINTS)) + " active",  ", ".join(_CONSTRAINTS)),
]

_df = pd.DataFrame(_rows, columns=["Parameter", "Default", "This Session", "Notes"])
_df["Changed"] = _df.apply(
    lambda r: "✔ reconfigured" if r["Default"] != r["This Session"] else "", axis=1
)

print("╔══════════════════════════════════════════════════════════════════════╗")
print("║   Phase 3 · Constrained MOBO · Simulation Configuration Summary      ║")
print("╚══════════════════════════════════════════════════════════════════════╝")
display(_df.to_string(index=False))
try:
    from IPython.display import display as _disp
    _disp(_df.style
         .set_caption("Phase 3 Constrained MOBO — Configuration Summary")
         .applymap(lambda v: "background-color:#ffeeba;font-weight:bold" if v == "✔ reconfigured" else "",
                   subset=["Changed"])
         .set_properties(**{"text-align": "left"})
         .hide(axis="index")
    )
except Exception:
    pass


In [ ]:
# Configuration & Parameter Bounds
from astra import Astra
A = Astra("../astra.in")
A.timeout = None
A.verbose = False
A.run()

init_parameters = [
    A["solenoid:maxb(1)"],
    A["quadrupole:q_grad(1)"],
    A["quadrupole:q_grad(2)"],
    A["cavity:phi(1)"],
    A["cavity:phi(2)"],
    A["cavity:phi(4)"],
]

ratio_list = [0.50, 0.50, 0.50, 0.10, 0.10, 0.10]
param_bounds_list = []
for val, r in zip(init_parameters, ratio_list):
    lower_val = val * (1 - r)
    upper_val = val * (1 + r)
    param_bounds_list.append([min(lower_val, upper_val), max(lower_val, upper_val)])

bounds = torch.tensor(param_bounds_list, dtype=torch.double).T
input_transform = Normalize(d=bounds.shape[1], bounds=bounds)
print("Parameter Bounds:
", bounds)


In [ ]:
# Define Constraint Functions on GP outcomes (Y shape ... x 9)
def c_sigma_x(Y): return Y[..., 3] - 1.0e-3
def c_sigma_y(Y): return Y[..., 4] - 1.0e-3
def c_sigma_xp(Y): return Y[..., 5] - 1.0e-3
def c_sigma_yp(Y): return Y[..., 6] - 1.0e-3
def c_sigma_z(Y): return Y[..., 7] - 1.0e-3
def c_E_min(Y): return 195e6 - Y[..., 8]
def c_E_max(Y): return Y[..., 8] - 205e6

CONSTRAINT_FUNCTIONS = [c_sigma_x, c_sigma_y, c_sigma_xp, c_sigma_yp, c_sigma_z, c_E_min, c_E_max]
objective_mapping = IdentityMCMultiOutputObjective(outcomes=[0, 1, 2])


In [ ]:
# Initialize Training Data
num_initial_samples = 16
num_workers = 12
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

sobol = torch.quasirandom.SobolEngine(dimension=bounds.shape[1], scramble=True, seed=seed)
samples = sobol.draw(num_initial_samples).to(dtype=torch.double)
train_X = bounds[0] + (bounds[1] - bounds[0]) * samples

executor = ThreadPoolExecutor(max_workers=num_workers)
print(f"Evaluating {num_initial_samples} initial samples...")
results = list(executor.map(evaluate_constrained_objective, train_X))

train_Y_list, train_Y_full_list, train_feas_list, initial_constraints_list = zip(*results)
train_Y = torch.stack(train_Y_list)
train_Y_full = torch.stack(train_Y_full_list)
train_feas_mask = torch.stack(train_feas_list)
train_constraints_list = list(initial_constraints_list)

print(f"Initial evaluation done. Feasible samples: {train_feas_mask.sum().item()} / {num_initial_samples}")


In [ ]:
# Constrained MOBO Optimization Loop
n_iterations = 20
q = 8
hypervolumes = []
run_dir = create_run_directory(base_dir="../results_notebooks/phase3_constrained")
checkpoint_file = os.path.join(run_dir, "gp_checkpoint", "constrained_mobo_checkpoint.pt")

print(f"Starting Phase 3 Constrained MOBO loop ({n_iterations} iterations, q={q})...")

for iteration in range(n_iterations):
    print(f"
--- Iteration {iteration+1}/{n_iterations} ---")
    # Fit 9 GPs on all data
    gps = [SingleTaskGP(train_X, train_Y_full[:, i:i+1], input_transform=input_transform, outcome_transform=Standardize(m=1)) for i in range(9)]
    model = ModelListGP(*gps)
    for m in model.models:
        mll = ExactMarginalLogLikelihood(m.likelihood, m)
        fit_gpytorch_mll(mll)

    feasible_Y = train_Y[train_feas_mask] if train_feas_mask.sum() > 0 else train_Y
    ref_point = compute_ref_point(feasible_Y)

    acq_func = qLogNoisyExpectedHypervolumeImprovement(
        model=model,
        ref_point=ref_point.tolist(),
        X_baseline=train_X,
        objective=objective_mapping,
        constraints=CONSTRAINT_FUNCTIONS,
        prune_baseline=True
    )

    candidates, _ = optimize_acqf(
        acq_function=acq_func,
        bounds=bounds,
        q=q,
        num_restarts=20,
        raw_samples=128,
        return_best_only=True
    )

    eval_res = list(executor.map(evaluate_constrained_objective, candidates))
    new_Y_list, new_Y_full_list, new_feas_list, new_constraints_tuples = zip(*eval_res)

    train_X = torch.cat([train_X, candidates])
    train_Y = torch.cat([train_Y, torch.stack(new_Y_list)])
    train_Y_full = torch.cat([train_Y_full, torch.stack(new_Y_full_list)])
    train_feas_mask = torch.cat([train_feas_mask, torch.stack(new_feas_list)])
    train_constraints_list.extend(list(new_constraints_tuples))

    feasible_Y_curr = train_Y[train_feas_mask]
    if feasible_Y_curr.shape[0] > 0:
        pareto_mask = is_non_dominated(feasible_Y_curr)
        current_hv = Hypervolume(ref_point=ref_point).compute(feasible_Y_curr[pareto_mask])
    else:
        current_hv = 0.0
    hypervolumes.append(current_hv)
    print(f"  Iteration {iteration+1} done | Feasible in batch: {torch.stack(new_feas_list).sum().item()}/{q} | HV: {current_hv:.4f}")

    save_checkpoint(iteration, train_X, train_Y, train_feas_mask, hypervolumes, train_constraints_list, "qLogNEHVI", checkpoint_file)
    save_results(train_X, train_Y, run_dir, hypervolumes, train_constraints_list)

executor.shutdown()
print("Optimization complete!")


## Visualization & Analysis

Comprehensive post-run plotting for Phase 3 Constrained MOBO.

> **Note**: Run the BO loop above before executing these cells.

In [ ]:
# ── Plotting imports ─────────────────────────────────────────────────────────
from mobo_linac.plotting import (
    plot_hypervolume_progress,
    plot_pareto_front,
    plot_pareto_front_3d,
    plot_objective_evolution,
    plot_best_so_far,
    plot_feasibility_rate,
    plot_constraint_diagnostics,
    plot_constraint_violins,
    plot_design_variable_heatmap,
    plot_parallel_coordinates,
    plot_gp_surrogate_slice,
)

results = []   # replace with EvaluationResult list if using mobo_linac API

_figures_dir = Path(run_dir) / "figures"
_figures_dir.mkdir(parents=True, exist_ok=True)
print(f"Saving figures to: {_figures_dir}")


### Hypervolume Progress

In [ ]:
hv_df = pd.DataFrame({"iteration": range(1, len(hypervolumes)+1),
                      "feasible_hypervolume": hypervolumes})
fig = plot_hypervolume_progress(hv_df,
                               output_path=_figures_dir / "hypervolume_progress.png")
plt.show()


### Objective Space — 2D & 3D Projections

In [ ]:
if results:
    fig = plot_pareto_front(results,
                           output_path=_figures_dir / "pareto_2d.png")
    plt.show()
    fig = plot_pareto_front_3d(results, elev=25, azim=45,
                               output_path=_figures_dir / "pareto_3d.png")
    plt.show()
else:
    # Fallback: plot directly from train_Y (model space, negated → physical)
    phys_Y = -train_Y.cpu().numpy()
    phys_Y_feas = phys_Y[train_feas_mask.cpu().numpy().astype(bool)]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    labels = [r'$\varepsilon_{n,x}$ [m·rad]',
              r'$\varepsilon_{n,y}$ [m·rad]',
              r'$\sigma_E$ [eV]']
    pairs = [(0,1), (2,0), (2,1)]
    for ax, (xi, yi) in zip(axes, pairs):
        ax.scatter(phys_Y[:,xi], phys_Y[:,yi], c='lightgray', s=20, alpha=0.4,
                   label='All valid')
        if len(phys_Y_feas):
            ax.scatter(phys_Y_feas[:,xi], phys_Y_feas[:,yi], c='steelblue',
                       s=35, alpha=0.85, label='Feasible')
        ax.set_xlabel(labels[xi], fontsize=11)
        ax.set_ylabel(labels[yi], fontsize=11)
        ax.grid(True, linestyle=':', alpha=0.6)
        ax.legend(fontsize=9)
    fig.suptitle('Objective Space — Phase 3 (Constrained)', fontsize=14)
    fig.tight_layout()
    plt.savefig(_figures_dir / 'pareto_2d_raw.png', dpi=200)
    plt.show()

    # 3D scatter
    from mpl_toolkits.mplot3d import Axes3D  # noqa
    fig3 = plt.figure(figsize=(9, 7))
    ax3 = fig3.add_subplot(111, projection='3d')
    if len(phys_Y_feas):
        sc = ax3.scatter(phys_Y_feas[:,0]*1e6, phys_Y_feas[:,1]*1e6,
                         phys_Y_feas[:,2]*1e-6,
                         c=phys_Y_feas[:,2]*1e-6, cmap='plasma', s=40, alpha=0.9)
        fig3.colorbar(sc, ax=ax3, shrink=0.6, label=r'$\sigma_E$ [MeV]')
    ax3.set_xlabel(r'$\varepsilon_{n,x}$ [mm·mrad]')
    ax3.set_ylabel(r'$\varepsilon_{n,y}$ [mm·mrad]')
    ax3.set_zlabel(r'$\sigma_E$ [MeV]')
    ax3.set_title('3D Pareto Front — Phase 3 (Feasible)')
    plt.savefig(_figures_dir / 'pareto_3d_raw.png', dpi=200)
    plt.show()


### Objective Evolution & Best-So-Far

In [ ]:
if results:
    fig = plot_objective_evolution(results,
                                  output_path=_figures_dir / "objective_evolution.png")
    plt.show()
    fig = plot_best_so_far(results,
                          output_path=_figures_dir / "best_so_far.png")
    plt.show()
else:
    phys_Y = -train_Y.cpu().numpy()
    feas_mask_np = train_feas_mask.cpu().numpy().astype(bool)
    labels = [r'$\varepsilon_{n,x}$', r'$\varepsilon_{n,y}$', r'$\sigma_E$']
    colors = ['steelblue', 'darkorange', 'seagreen']

    fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
    for i, (ax, lbl, col) in enumerate(zip(axes, labels, colors)):
        vals = phys_Y[:, i]
        ax.plot(vals, color=col, linewidth=1.4, alpha=0.7)
        ax.scatter(range(len(vals)), vals,
                   c=['steelblue' if f else 'lightgray' for f in feas_mask_np],
                   s=18, zorder=3)
        ax.plot(pd.Series(vals).cummin(), color=col, linestyle='--',
                linewidth=2, label='Best so far')
        ax.set_ylabel(lbl, fontsize=11)
        ax.grid(True, linestyle=':', alpha=0.6)
        ax.legend(fontsize=9)
    axes[-1].set_xlabel('Evaluation Index')
    fig.suptitle('Objective Evolution — Phase 3', fontsize=14)
    fig.tight_layout()
    plt.savefig(_figures_dir / 'objective_evolution_raw.png', dpi=200)
    plt.show()


### Feasibility Rate

In [ ]:
if results:
    fig = plot_feasibility_rate(results, window=10,
                               output_path=_figures_dir / "feasibility_rate.png")
    plt.show()
else:
    feas_flag = train_feas_mask.cpu().numpy().astype(float)
    cumrate = pd.Series(feas_flag).expanding().mean() * 100
    rollrate = pd.Series(feas_flag).rolling(10, min_periods=1).mean() * 100
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(cumrate, color='steelblue', linewidth=2, label='Cumulative')
    ax.plot(rollrate, color='darkorange', linestyle='--', linewidth=1.8,
            label='Rolling 10-eval')
    ax.fill_between(range(len(cumrate)), cumrate, alpha=0.12, color='steelblue')
    ax.set_ylim(0, 105)
    ax.set_xlabel('Evaluation Index')
    ax.set_ylabel('Feasibility Rate [%]')
    ax.set_title('Beam Feasibility Rate — Phase 3')
    ax.legend(fontsize=10)
    ax.grid(True, linestyle=':', alpha=0.6)
    plt.savefig(_figures_dir / 'feasibility_rate_raw.png', dpi=200)
    plt.show()


### Constraint Diagnostics & Violin Plots

In [ ]:
if results:
    fig = plot_constraint_diagnostics(results,
                                     output_path=_figures_dir / "constraint_diagnostics.png")
    plt.show()
    fig = plot_constraint_violins(results,
                                 output_path=_figures_dir / "constraint_violins.png")
    plt.show()
else:
    # Legacy — use original constraint list from Phase 3 loop
    plot_all_constraints(train_constraints_list, train_feas_mask)


### Design Variable Correlation & Parallel Coordinates

In [ ]:
if results:
    fig = plot_design_variable_heatmap(results, feasible_only=True,
                                      output_path=_figures_dir / "design_var_heatmap.png")
    plt.show()
    fig = plot_parallel_coordinates(results, color_by='norm_emit_x_m_rad',
                                   feasible_only=True, n_lines=200,
                                   output_path=_figures_dir / "parallel_coordinates.png")
    plt.show()
else:
    # Parallel coordinates directly from train_X tensor
    import matplotlib.cm as _cm
    import matplotlib.colors as _mcol
    feas_mask_np = train_feas_mask.cpu().numpy().astype(bool)
    X_np = train_X.cpu().numpy()
    Y_np = -train_Y.cpu().numpy()
    X_f = X_np[feas_mask_np]
    Y_f = Y_np[feas_mask_np]
    c_vals = Y_f[:, 0]
    cmap = _cm.plasma
    cnorm = _mcol.Normalize(c_vals.min(), c_vals.max())
    # normalise columns to [0,1]
    X_norm = (X_f - X_f.min(0)) / (X_f.max(0) - X_f.min(0) + 1e-30)
    n_vars = X_norm.shape[1]
    fig, ax = plt.subplots(figsize=(12, 6))
    for row_i, row in enumerate(X_norm):
        rgba = cmap(cnorm(c_vals[row_i]))
        ax.plot(range(n_vars), row, color=rgba, alpha=0.45, linewidth=0.9)
    ax.set_xticks(range(n_vars))
    var_lbls = [r'$B_{sol}$', r'$G_{q1}$', r'$G_{q2}$',
                r'$\phi_{gun}$', r'$\phi_{12}$', r'$\phi_{34}$']
    ax.set_xticklabels(var_lbls, fontsize=11)
    import matplotlib.cm as _cm2
    sm = _cm2.ScalarMappable(cmap=cmap, norm=cnorm)
    fig.colorbar(sm, ax=ax, label=r'$\varepsilon_{n,x}$ [m·rad]', shrink=0.8)
    ax.set_ylabel('Normalised Value')
    ax.set_title('Parallel Coordinates — Feasible (Phase 3)')
    ax.grid(True, axis='x', linestyle=':', alpha=0.5)
    plt.savefig(_figures_dir / 'parallel_coordinates_raw.png', dpi=200)
    plt.show()


### GP Surrogate Posterior Slice

Visualise uncertainty learned by each of the 9 GP models along one design-variable axis.

In [ ]:
# Requires `model` (ModelListGP, 9 GPs) from the last iteration of the BO loop.
# Uncomment the block below after running the loop.
#
# from mobo_linac.plotting import plot_gp_surrogate_slice
# import torch
#
# fixed_x = train_X.mean(dim=0)   # use mean of all evaluated points
# obj_names = [r'$\varepsilon_x$', r'$\varepsilon_y$', r'$\sigma_E$',
#              r'$\sigma_x$', r'$\sigma_y$', r'$\sigma_{xp}$',
#              r'$\sigma_{yp}$', r'$\sigma_z$', r'$E_{kin}$']
# for dim in range(6):
#     for obj_idx in range(9):
#         fig = plot_gp_surrogate_slice(
#             model, bounds, fixed_x,
#             dim=dim, obj_idx=obj_idx,
#             output_path=_figures_dir / f'gp_slice_dim{dim}_obj{obj_idx}.png',
#         )
#         plt.show()
print('GP surrogate slice: uncomment the block above and supply the fitted model.')
